<a href="https://colab.research.google.com/github/abyanrizz/practicallinearalgebra/blob/main/Chapter_15_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#chapter 15
Eigendecomposition and SVD are gems that linear algebra has bestowed upon
modern human civilization. Their importance in modern applied mathematics can‐
not be understated, and their applications are uncountable and spread across myriad
disciplines.
In this chapter, I will highlight three applications that you are likely to come across
in data science and related fields. My main goal is to show you that seemingly
complicated data science and machine learning techniques are actually quite sensible
and easily understood, once you’ve learned the linear algebra topics in this book.
PCA Using Eigendecomposition and SVD
The purpose of PCA is to find a set of basis vectors for a dataset that point in the
direction that maximizes covariation across the variables.
Imagine that an N-D dataset exists in an N-D space, with each data point being a
coordinate in that space. This is sensible when you think about storing the data in a
matrix with N observations (each row is an observation) of M features (each column
is a feature, also called variable or measurement); the data live in R

M and comprise N

vectors or coordinates.
An example in 2D is shown in Figure 15-1. The left-side panel shows the data in its
original data space, in which each variable provides a basis vector for the data. Clearly
the two variables (the x- and y-axes) are related to each other, and clearly there is a
direction in the data that captures that relation better than either of the feature basis
vectors.

255

Figure 15-1. Graphical overview of PCA in 2D
The goal of PCA is to find a new set of basis vectors such that the linear relationships
across the variables are maximally aligned with the basis vectors—that’s what the
right-side panel of Figure 15-1 shows. Importantly, PCA has the constraint that
the new basis vectors are orthogonal rotations of the original basis vectors. In the
exercises, you will see the implications of this constraint.
In the next section, I will introduce the math and procedures for computing PCA; in
the exercises, you will have the opportunity to implement PCA using eigendecompo‐
sition and SVD, and compare your results against Python’s implementation of PCA.
The Math of PCA
PCA combines the statistical concept of variance with the linear algebra concept of
linear weighted combination. Variance, as you know, is a measure of the dispersion of
a dataset around its average value. PCA makes the assumption that variance is good,
and directions in the data space that have more variance are more important (a.k.a.
“variance = relevance”).
But in PCA, we’re not just interested in the variance within one variable; instead, we
want to find the linear weighted combination across all variables that maximizes var‐
iance of that component (a component is a linear weighted combination of variables).
Let’s write this down in math. Matrix X is our data matrix (a tall full column-rank
matrix of observations by features), and w is the vector of weights. Our goal in PCA
is to find the set of weights in w such that Xw has maximal variance. Variance is a
scalar, so we can write that down as:

256 | Chapter 15: Eigendecomposition and SVD Applications

1 The online code demonstrates this point.
λ = ∥ Xw ∥ 2
The squared vector norm is actually the same thing as variance when the data is
mean-centered (that is, each data variable has a mean of zero);1

I’ve omitted a scaling
factor of 1/(N − 1), because it does not affect the solution to our optimization goal.
The problem with that equation is you can simply set w to be HUGE numbers; the
bigger the weights, the larger the variance. The solution is to scale the norm of the
weighted combination of data variables by the norm of the weights:
λ =
∥ Xw ∥ 2
∥ w ∥ 2
Now we have a ratio of two vector norms. We can expand those norms into dot
products to gain some insight into the equation:
λ =
w
TX
TXw
w
Tw
C = X
TX
λ =
w
TCw
w
Tw
We’ve now discovered that the solution to PCA is the same as the solution to finding
the directional vector that maximizes the normalized quadratic form (the vector
norm is the normalization term) of the data covariance matrix.
That’s all fine, but how do we actually find the elements in vector w that maximize λ?
The linear algebra approach here is to consider not just a single vector solution but
an entire set of solutions. Thus, we rewrite the equation using matrix W instead of
vector w. That would give a matrix in the denominator, which is not a valid operation
in linear algebra; therefore, we multiply by the inverse:
Λ = WTW
−1
WTCW

PCA Using Eigendecomposition and SVD | 257

From here, we apply some algebra and see what happens:
Λ = WTW
−1
WTCW
Λ = W−1W−TWTCW
Λ = W−1CW
WΛ = CW
Remarkably, we’ve discovered that the solution to PCA is to perform an eigendecom‐
position on the data covariance matrix. The eigenvectors are the weights for the data
variables, and their corresponding eigenvalues are the variances of the data along
each direction (each column of W).
Because covariance matrices are symmetric, their eigenvectors—and therefore princi‐
pal components—are orthogonal. This has important implications for the appropri‐
ateness of PCA for data analysis, which you will discover in the exercises.

PCA Proof

I’m going to write out the proof that eigendecomposition solves the PCA optimiza‐
tion goal. If you are not familiar with calculus and Lagrange multipliers, then please
feel free to skip this box; I include it here in the interest of completeness, not because
you need to understand it to solve the exercises or use PCA in practice.
Our goal is to maximize w

TCw subject to the constraint that w

Tw = 1. We can

express this optimization using a Lagrange multiplier:
L w, λ = w
TCw − λ w
Tw − 1

0 = d
dw w
TCw − λ w
Tw − 1

0 = Cw − λw
Cw = λw
Briefly, the idea is that we use the Lagrange multiplier to balance the optimization
with the constraint, take the derivative with respect to the weights vector, set the
derivative to zero, differentiate with respect to w, and discover that w is an eigenvec‐
tor of the covariance matrix.

258 | Chapter 15: Eigendecomposition and SVD Applications

2 In Exercise 15-3, you will also learn how to implement PCA using the Python scikit-learn library.
The Steps to Perform a PCA
With the math out of the way, here are the steps to implement a PCA:2
1. Compute the covariance matrix of the data. The resulting covariance matrix will
be features-by-features. Each feature in the data must be mean-centered prior to
computing covariance.
2. Take the eigendecomposition of that covariance matrix.
3. Sort the eigenvalues descending by magnitude, and sort the eigenvectors accord‐
ingly. Eigenvalues of the PCA are sometimes called latent factor scores.
4. Compute the “component scores” as the weighted combination of all data fea‐
tures, where the eigenvector provides the weights. The eigenvector associated
with the largest eigenvalue is the “most important” component, meaning the one
with the largest variance.
5. Convert the eigenvalues to percent variance explained to facilitate interpretation.
PCA via SVD
PCA can equivalently be performed via eigendecomposition as previously described
or via SVD. There are two ways to perform a PCA using SVD:
• Take the SVD of the covariance matrix. The procedure is identical to that previ‐
ously described, because SVD and eigendecomposition are the same decomposi‐
tion for covariance matrices.
• Take the SVD of the data matrix directly. In this case, the right singular vec‐
tors (matrix V) are equivalent to the eigenvectors of the covariance matrix

(it would be the left singular vectors if the data matrix is stored as features-by-
observations). The data must be mean-centered before computing the SVD.

The square root of the singular values is equivalent to the eigenvalues of the
covariance matrix.
Should you use eigendecomposition or SVD to perform a PCA? You might think
that SVD is easier because it does not require the covariance matrix. That’s true
for relatively small and clean datasets. But larger or more complicated datasets may
require data selection or may be too memory intensive to take the SVD of the
entire data matrix. In these cases, computing the covariance matrix first can increase
analysis flexibility. But the choice of eigendecomposition versus SVD is often a matter
of personal preference.

PCA Using Eigendecomposition and SVD | 259

3 Indeed, linear discriminant analysis is also called Fisher’s discriminant analysis.
Linear Discriminant Analysis
Linear discriminant analysis (LDA) is a multivariate classification technique that is
often used in machine learning and statistics. It was initially developed by Ronald
Fisher,3
who is often considered the “grandfather” of statistics for his numerous and
important contributions to the mathematical foundations of statistics.
The goal of LDA is to find a direction in the data space that maximally separates
categories of data. An example problem dataset is shown in graph A in Figure 15-2.
It is visually obvious that the two categories are separable, but they are not separable
on either of the data axes alone—that is clear from visual inspection of the marginal
distributions.
Enter LDA. LDA will find basis vectors in the data space that maximally separate
the two categories. Graph B in Figure 15-2 shows the same data but in the LDA
space. Now the classification is simple—observations with negative values on axis-1
are labeled category “0” and any observations with positive values on axis 1 are
labeled category “1.” The data is completely inseparable on axis 2, indicating that one
dimension is sufficient for accurate categorization in this dataset.

Figure 15-2. Example 2D problem for LDA

260 | Chapter 15: Eigendecomposition and SVD Applications

4 I won’t go through the calculus-laden proof, but it’s just a minor variant of the proof given in the PCA section.
Sounds great, right? But how does such a marvel of mathematics work? It’s actually
fairly straightforward and based on generalized eigendecomposition, which you
learned about toward the end of Chapter 13.
Let me begin with the objective function: our goal is to find a set of weights such
that the weighted combination of variables maximally separates the categories. That
objective function can be written similarly as with the PCA objective function:
λ =
∥ XBw ∥ 2
∥ XWw ∥ 2
In English: we want to find a set of feature weights w that maximizes the ratio of the
variance of data feature XB, to the variance of data feature XW. Notice that the same
weights are applied to all data observations. (I’ll write more about data features B and
W after discussing the math.)
The linear algebra solution comes from following a similar argument as described in
the PCA section. First, expand ∥ XBw ∥ 2
to w
TXB
TXBw and express this as w
TCBw;
second, consider a set of solutions instead of one solution; third, replace the division
with multiplication of the inverse; and finally, do some algebra and see what happens:
Λ = WTCWW
−1
WTCBW

Λ = W−1CW
−1W−TWTCBW
Λ = W−1CW
−1CBW
WΛ = CW
−1CBW
CWWΛ = CBW
In other words, the solution to LDA comes from a generalized eigendecomposition
on two covariance matrices. The eigenvectors are the weights, and the generalized
eigenvalues are the variance ratios of each component.4
With the math out of the way, which data features are used to construct XB and XW?
Well, there are different ways of implementing that formula, depending on the nature
of the problem and the specific goal of the analysis. But in a typical LDA model,
the XB comes from the between-category covariance while the XW comes from the
within-category covariance.

Linear Discriminant Analysis | 261

The within-category covariance is simply the average of the covariances of the data
samples within each class. The between-category covariance comes from creating
a new data matrix comprising the feature averages within each class. I will walk
you through the procedure in the exercises. If you are familiar with statistics, then
you’ll recognize this formulation as analogous to the ratio of between-group to
within-group sum of squared errors in ANOVA models.
Two final comments: The eigenvectors of a generalized eigendecomposition are not
constrained to be orthogonal. That’s because CW

−1CB is generally not a symmetric
matrix even though the two covariance matrices are themselves symmetric. Nonsym‐
metric matrices do not have the orthogonal-eigenvector constraint. You’ll see this in
the exercises.
Finally, LDA will always find a linear solution (duh, that’s right in the name LDA),
even if the data is not linearly separable. Nonlinear separation would require a trans‐
formation of the data or the use of a nonlinear categorization method like artificial
neural networks. LDA will still work in the sense of producing a result; it’s up to you
as the data scientist to determine whether that result is appropriate and interpretable
for a given problem.
Low-Rank Approximations via SVD
I explained the concept of low-rank approximations in the previous chapter (e.g.,
Exercise 14-5). The idea is to take the SVD of a data matrix or image, and then
reconstruct that data matrix using some subset of SVD components.
You can achieve this by setting selected σs to equal zero or by creating new SVD
matrices that are rectangular, with the to-be-rejected vectors and singular values
removed. This second approach is preferred because it reduces the sizes of the data
to be stored, as you will see in the exercises. In this way, the SVD can be used to
compress data down to a smaller size.

Do Computers Use SVD to Compress Images?

The short answer is no, they don’t.

Algorithms underlying common image compression formats like JPG involve block-
wise compression that incorporates principles of human perception (including, for

example, how we perceive contrast and spatial frequencies), allowing them to achieve
better results with less computational power, compared to taking one SVD of the
entire image.
Nonetheless, the principle is the same as with SVD-based compression: identify a
small number of basis vectors that preserves the important features of an image such

262 | Chapter 15: Eigendecomposition and SVD Applications

that the low-rank reconstruction is an accurate approximation of the full-resolution
original.
That said, SVD for data compression is commonly used in other areas of science,
including biomedical imaging.
SVD for Denoising
Denoising via SVD is simply an application of low-rank approximation. The only
difference is that SVD components are selected for exclusion based on them repre‐
senting noise as opposed to making small contributions to the data matrix.
The to-be-removed components might be layers associated with the smallest singular
values—that would be the case for low-amplitude noise associated with small equipment
imperfections. But larger sources of noise that have a stronger impact on the data might
have larger singular values. These noise components can be identified by an algorithm
based on their characteristics or by visual inspection. In the exercises, you will see an
example of using SVD to separate a source of noise that was added to an image.
Summary
You’ve made it to the end of the book (except for the exercises below)! Congrats! Take
a moment to be proud of yourself and your commitment to learning and investing in
your brain (it is, after all, your most precious resource). I am proud of you, and if we
would meet in person, I’d give you a high five, fist bump, elbow tap, or whatever is
socially/medically appropriate at the time.
I hope you feel that this chapter helped you see the incredible importance of eigen‐
decomposition and singular value decomposition to applications in statistics and
machine learning. Here’s a summary of the key points that I am contractually obliga‐
ted to include:
• The goal of PCA is to find a set of weights such that the linear weighted combi‐
nation of data features has maximal variance. That goal reflects the assumption
underlying PCA, which is that “variance equals relevance.”
• PCA is implemented as the eigendecomposition of a data covariance matrix.
The eigenvectors are the feature weightings, and the eigenvalues can be scaled to
encode percent variance accounted for by each component (a component is the
linear weighted combination).
• PCA can be equivalently implemented using the SVD of the covariance matrix or
the data matrix.

Summary | 263

5 Data citation: Akbilgic, Oguz. (2013). Istanbul Stock Exchange. UCI Machine Learning Repository. Data
source website: https://archive-beta.ics.uci.edu/ml/datasets/istanbul+stock+exchange.
• Linear discriminant analysis (LDA) is used for linear categorization of multivari‐
able data. It can be seen as an extension of PCA: whereas PCA maximizes
variance, LDA maximizes the ratio of variances between two data features.
• LDA is implemented as a generalized eigendecomposition on two covariance
matrices that are formed from two different data features. The two data features
are often the between-class covariance (to maximize) and the within-class cova‐
riance (to minimize).
• Low-rank approximations involve reproducing a matrix from a subset of singular
vectors/values and are used for data compression and denoising.
• For data compression, the components associated with the smallest singular val‐
ues are removed; for data denoising, components that capture noise or artifacts
are removed (their corresponding singular values could be small or large).
Exercises
PCA
I love Turkish coffee. It’s made with very finely ground beans and no filter. The whole
ritual of making it and drinking it is wonderful. And if you drink it with a Turk,
perhaps you can have your fortune read.
This exercise is not about Turkish coffee, but it is about doing a PCA on a dataset5
that contains time series data from the Istanbul stock exchange, along with stock
exchange data from several other stock indices in different countries. We could use
this dataset to ask, for example, whether the international stock exchanges are driven
by one common factor, or whether different countries have independent financial
markets.
Exercise 15-1.
Before performing a PCA, import and inspect the data. I made several plots of the
data shown in Figure 15-3; you are welcome to reproduce these plots and/or use
different methods to explore the data.

264 | Chapter 15: Eigendecomposition and SVD Applications

Figure 15-3. Some investigations of the international stock exchange dataset
Now for the PCA. Implement the PCA using the five steps outlined earlier in this
chapter. Visualize the results as in Figure 15-4. Use code to demonstrate several
features of PCA:
1. The variance of the component time series (using np.var) equals the eigenvalue
associated with that component. You can see the results for the first two compo‐
nents here:

In [ ]:
Variance of first two components:
[0.0013006 0.00028585]
First two eigenvalues:
[0.0013006 0.00028585]